<a href="https://colab.research.google.com/github/lack-of-sleep/2023-1-UPP/blob/main/apling_hw01_GloVe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
t_fname = 'Alice.txt'
d_fname = 'Analogy.txt'
m_fname = 'GloVe.bin'
m_tname = 'GloVe.txt'

In [ ]:
dim = 300
lr = 0.05

In [ ]:
import nltk

In [ ]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
file = open(t_fname, 'r', encoding='utf8')
text = file.read()
text = text.replace('\n', ' ')
text = text.replace('\t', ' ')
text = text.replace(' ', ' ')
text = text.lower()


In [ ]:
from nltk.tokenize import sent_tokenize
from nltk.tokenize import word_tokenize


In [ ]:
sentences = sent_tokenize(text)
for i in range(0, len(sentences)):
  temp = sentences[i]
  sentences[i] = word_tokenize(temp)

In [ ]:
! pip install glove_python_binary --verbose
#버전 차이로 설치안됨 오류남

Using pip 23.0.1 from /usr/local/lib/python3.9/dist-packages/pip (python 3.9)
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
ERROR: Could not find a version that satisfies the requirement glove_python_binary (from versions: none)
ERROR: No matching distribution found for glove_python_binary


In [ ]:
import gensim
from glove import Corpus, Glove


ModuleNotFoundError: ignored

In [ ]:
corpus = Corpus()
corpus.fit(sentences, window=2)
model = Glove(no_components=dim, learning_rate=lr)
model.fit(corpus.matrix, epochs=5, no_threads=2, verbose=False)
print(model)

In [ ]:
model.add_dictionary(corpus.dictionary)
words = list(model.dictionary)
print(words)

In [ ]:
print(model.word_vectors[model.dictionary['the']])


In [ ]:
import numpy as np
import pandas as pd


In [ ]:
model.save(m_fname)



In [ ]:
nrows = len(words)
ncols = 300
word2vec_df = pd.DataFrame(np.random.randn(nrows, ncols))


In [ ]:
for i in range(0, len(words)):
  word = words[i]
  feat = model.word_vectors[model.dictionary[word]]
  for j in range(0, ncols):
    word2vec_df[j][i] = feat[j]
  if (((i + 1) % 100) == 0):
      print(str(i + 1) + '/' + str(nrows) + ' was completed.')
word2vec_df.insert(0,'',words,True)
word2vec_df = word2vec_df.sort_values(word2vec_df.columns[0])


In [ ]:
o_file = open(m_tname, 'w', newline='')
o_file.write(word2vec_df.to_csv(sep="\t", header=False, index=False))
o_file.close()


In [ ]:
model = Glove.load(m_fname)
print(model)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from glove import Glove
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE


In [ ]:
vocab = list(model.dictionary)
vocab_list = []
for i in range(0, len(vocab)):
  vocab_list.append(model.word_vectors[model.dictionary[vocab[i]]])

In [ ]:
size = len(vocab_list)
vocab_list_show = vocab_list[:size]
vocab_show = vocab[:size]


In [ ]:
def show_pca():
  pca = PCA(n_components=2)
  pca.fit(vocab_list_show)
  vocab_list_pca = pca.transform(vocab_list_show)

  plt.figure(figsize=(15, 10))
  plt.xlim(vocab_list_pca[:, 0].min(), vocab_list_pca[:, 0].max())
  plt.ylim(vocab_list_pca[:, 1].min(), vocab_list_pca[:, 1].max())
  for i in range(len(vocab_list_show)):
    plt.text(vocab_list_pca[i, 0], vocab_list_pca[i, 1], str(vocab_show[i]),fontdict={'weight': 'bold', 'size': 9})
  plt.xlabel("1st Principal Component")
  plt.ylabel("2nd Principal Component")
  plt.show()

In [ ]:
show_pca()


In [ ]:
def show_tsne():
  tsne = TSNE(n_components=2)
  vocab_list = tsne.fit_transform(vocab_list_show)

  df = pd.DataFrame(vocab_list, index=vocab_show, columns=['x', 'y'])
  fig = plt.figure()
  fig.set_size_inches(15, 10)
  ax = fig.add_subplot(1, 1, 1)
  ax.scatter(df['x'], df['y'])

  plt.xlabel("t-SNE Characteristic 0")
  plt.ylabel("t-SNE Characteristic 1")
  plt.show()
  return df

In [ ]:
show_tsne()


In [ ]:
d_df = pd.DataFrame(columns=['Word1', 'Word2', 'Word3', 'Word4'])
i_file = open(d_fname, 'r', encoding='utf8')
text_cont = i_file.readlines()
d_fnum = len(text_cont)
for i in range(0,d_fnum):
  text_line = text_cont[i]
  token = text_line.split()
  d_df = d_df.append(pd.DataFrame([[token[0], token[1], token[2], token[3]]], columns=['Word1', 'Word2', 'Word3', 'Word4']), ignore_index=True)
i_file.close()


In [ ]:
from gensim.models import KeyedVectors
from gensim.scripts.glove2word2vec import glove2word2vec

In [ ]:
glove_input_file = m_tname
word2vec_output_file = m_tname + '.word2vec'

In [ ]:
glove2word2vec(glove_input_file, word2vec_output_file)

In [ ]:
filename = word2vec_output_file
model = KeyedVectors.load_word2vec_format(filename, binary=False, limit=5000)

In [ ]:
m_item = 0
c_item = 0
scores = 0

In [ ]:
for i in range(0, d_fnum):
  word1 = d_df.iloc[i, 0]
  word2 = d_df.iloc[i, 1]
  word3 = d_df.iloc[i, 2]
  word4 = d_df.iloc[i, 3]

  if ((word1 in words) & (word2 in words) & (word3 in words)):
    m_item = m_item + 1
    s_word = model.most_similar(positive=[word1, word3], negative=[word2], topn=5)
    for j in range(0, 5):
      if (s_word[j][0]==word4):
        c_item = c_item + 1
        scores = scores + 5 - j

In [ ]:
value = round(m_item/d_fnum*100, 3)
print('The percentage of the matched analogy: ' + str(value))

In [ ]:
value = round(c_item/m_item*100, 3)
print('The percentage of the correct match: ' + str(value))

In [ ]:
value = round((scores/(m_item*5))*100, 3)
print('The similarity scores of the matched analogy: ' + str(value))


In [ ]:
value = round((scores/(c_item*5))*100, 3)
print('The similarity scores of the correct match: ' + str(value))

In [ ]:
! wget http://nlp.stanford.edu/data/glove.6B.zip
! unzip -q glove.6B.zip

In [ ]:
from gensim.models import KeyedVectors
from gensim.scripts.glove2word2vec import glove2word2vec

In [ ]:
glove_input_file = 'glove.6B.300d.txt'
word2vec_output_file = 'glove.6B.300d.txt.word2vec'

In [ ]:
glove2word2vec(glove_input_file, word2vec_output_file)

In [ ]:
filename = word2vec_output_file
model = KeyedVectors.load_word2vec_format(filename, binary=False, limit=5000)

In [ ]:
words = list(model.wv.vocab)
print(words)

In [ ]:
print(model['the'])

In [ ]:
model.most_similar(positive=['king', 'woman'], negative=['man'], topn=1)


In [ ]:
model.similarity('go','run')

In [ ]:
import torch
import torchtext

In [ ]:
glove = torchtext.vocab.GloVe(name="6B", dim=50)


In [ ]:
x = glove['dog']
y = glove['cat']
torch.norm(y - x)

In [ ]:
x = glove['dog']
y = glove['cat']
torch.cosine_similarity(glove['dog'].unsqueeze(0), glove['cat'].unsqueeze(0))